# Diagnóstico do Poder Preditivo das Features

A primeira rodada de modelagem apresentou capacidade discriminatória limitada.

A Random Forest obteve o melhor ROC-AUC na validação (0,5769), porém sua acurácia permaneceu próxima ao baseline ingênuo.

Antes de aumentar a complexidade dos modelos ou realizar otimização de hiperparâmetros, este notebook investiga possíveis causas para o baixo desempenho.

A análise busca verificar:

- a integridade da base analítica e dos joins;
- a ambiguidade existente dentro dos contextos município + rede;
- o poder discriminatório e a variabilidade das features disponíveis;
- possíveis diferenças de distribuição entre treino e validação;
- a necessidade de uma nova etapa de engenharia e enriquecimento de features.

O conjunto de teste temporal de 2024 permanece fora desta análise.

# 1. Configuração
Nesta etapa são importadas as bibliotecas utilizadas no notebook

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import QuantileDiscretizer

## Carregamento das Bases

A análise utiliza exclusivamente os conjuntos de treino e validação de 2023.

O teste temporal de 2024 permanece reservado para a avaliação final do modelo.

In [0]:
df_gold = spark.table("workspace.gold.ml_alfabetizacao_aluno")

df_split = spark.table("workspace.gold.ml_split_modelagem")

df_modelagem = df_gold.join(df_split, on=["ano", "id_municipio"], how="inner")

df_desenvolvimento = df_modelagem.filter(F.col("split").isin("treino", "validacao"))

# 2. Auditoria estrutural

## 2.1 Unicidade do mapa de splits

In [0]:
duplicidades_split = (
    df_split.groupBy("ano", "id_municipio").count().filter(F.col("count") > 1)
)

print("Chaves duplicadas no mapa de split:", duplicidades_split.count())

## 2.2 Integridade dos alunos após o join

In [0]:
display(
    df_desenvolvimento.groupBy("split").agg(
        F.count("*").alias("qtd_linhas"),
        F.countDistinct(F.struct("ano", "id_aluno")).alias("qtd_alunos_unicos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
    )
)

# 3. Ambiguidade das Features

As features utilizadas atualmente possuem predominantemente granularidade municipal ou município + rede, enquanto o target possui granularidade
individual.

Consequentemente, diferentes alunos podem possuir exatamente o mesmo conjunto de características contextuais e, ainda assim, apresentar targets distintos.

Para mensurar essa limitação é calculado um teto empírico contextual.

Seu objetivo é exclusivamente diagnosticar a quantidade de ambiguidade existente entre alunos que compartilham o mesmo contexto observável.

## 3.1 Acertos Máximos

In [0]:
df_contextos_target = (
    df_desenvolvimento.groupBy("split", "id_municipio", "rede")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.sum(F.when(F.col("alfabetizado") == 0, 1).otherwise(0)).alias("qtd_0"),
        F.sum(F.when(F.col("alfabetizado") == 1, 1).otherwise(0)).alias("qtd_1"),
    )
    .withColumn("acertos_max_contexto", F.greatest(F.col("qtd_0"), F.col("qtd_1")))
    .withColumn(
        "contexto_misto", ((F.col("qtd_0") > 0) & (F.col("qtd_1") > 0)).cast("int")
    )
)

In [0]:
display(df_contextos_target.orderBy(F.desc("qtd_alunos")))

In [0]:
display(
    df_contextos_target.groupBy("split").agg(
        F.sum("qtd_alunos").alias("qtd_alunos"),
        F.sum("acertos_max_contexto").alias("acertos_maximos"),
        F.round(F.sum("acertos_max_contexto") / F.sum("qtd_alunos"), 4).alias(
            "accuracy_max_contextual"
        ),
        F.round(F.avg("contexto_misto") * 100, 2).alias(
            "pct_contextos_com_classes_mistas"
        ),
    )
)

## 3.2 Pureza dos Contextos

In [0]:
df_contextos_target = df_contextos_target.withColumn(
    "taxa_classe_majoritaria",
    F.greatest(F.col("qtd_0"), F.col("qtd_1")) / F.col("qtd_alunos"),
)

In [0]:
display(
    df_contextos_target.groupBy("split").agg(
        F.expr("percentile_approx(taxa_classe_majoritaria, 0.10)").alias("p10"),
        F.expr("percentile_approx(taxa_classe_majoritaria, 0.25)").alias("p25"),
        F.expr("percentile_approx(taxa_classe_majoritaria, 0.50)").alias("mediana"),
        F.expr("percentile_approx(taxa_classe_majoritaria, 0.75)").alias("p75"),
        F.expr("percentile_approx(taxa_classe_majoritaria, 0.90)").alias("p90"),
    )
)

## 3.3 Análise Contexto Ponderando

In [0]:
display(
    df_contextos_target.groupBy("split").agg(
        F.round(
            100
            * F.sum(
                F.when(F.col("contexto_misto") == 1, F.col("qtd_alunos")).otherwise(0)
            )
            / F.sum("qtd_alunos"),
            2,
        ).alias("pct_alunos_contextos_mistos"),
        F.round(
            100
            * F.sum(
                F.when(
                    F.col("taxa_classe_majoritaria") <= 0.55, F.col("qtd_alunos")
                ).otherwise(0)
            )
            / F.sum("qtd_alunos"),
            2,
        ).alias("pct_alunos_contextos_ate_55"),
        F.round(
            100
            * F.sum(
                F.when(
                    F.col("taxa_classe_majoritaria") <= 0.60, F.col("qtd_alunos")
                ).otherwise(0)
            )
            / F.sum("qtd_alunos"),
            2,
        ).alias("pct_alunos_contextos_ate_60"),
        F.round(
            100
            * F.sum(
                F.when(
                    F.col("taxa_classe_majoritaria") <= 0.70, F.col("qtd_alunos")
                ).otherwise(0)
            )
            / F.sum("qtd_alunos"),
            2,
        ).alias("pct_alunos_contextos_ate_70"),
    )
)

## Diagnóstico da Granularidade Contextual

A auditoria estrutural não identificou duplicidades no mapa de splits ou multiplicação de registros durante a reconstrução da base, indicando consistência
na construção do conjunto de modelagem.

Entretanto, foi identificada uma limitação relevante relacionada à granularidade das features.

Mesmo em um cenário hipotético em que fosse conhecida a classe majoritária de cada contexto, a acurácia máxima contextual seria de aproximadamente 64,06%, comparada ao baseline de 58,20%.

A mediana da proporção da classe majoritária nos contextos foi de 66,89%, evidenciando elevada heterogeneidade entre alunos que compartilham as mesmas características contextuais.

Esse resultado sugere que as variáveis atualmente disponíveis possuem capacidade limitada para discriminar individualmente alunos alfabetizados e não alfabetizados.

O resultado não representa um limite matemático do problema, mas funciona como um diagnóstico da quantidade de informação discriminatória disponível na granularidade atual.

# 4. Variabilidade das Features

Após identificar elevada ambiguidade entre alunos que compartilham o mesmo contexto município + rede, é avaliada a variabilidade das features utilizadas na primeira rodada de modelagem.

Variáveis com baixa dispersão ou poucos valores distintos podem contribuir pouco para a separação entre os contextos, mesmo quando conceitualmente relevantes.

## 4.1 Variabilidade e Percentis

In [0]:
features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
]

In [0]:
resultados_variabilidade = []

for coluna in features_numericas:
    estatisticas = (
        df_desenvolvimento.filter(F.col("split") == "treino")
        .agg(
            F.min(coluna).alias("min"),
            F.expr(f"percentile_approx({coluna}, 0.05)").alias("p05"),
            F.expr(f"percentile_approx({coluna}, 0.25)").alias("p25"),
            F.expr(f"percentile_approx({coluna}, 0.50)").alias("mediana"),
            F.expr(f"percentile_approx({coluna}, 0.75)").alias("p75"),
            F.expr(f"percentile_approx({coluna}, 0.95)").alias("p95"),
            F.max(coluna).alias("max"),
            F.stddev(coluna).alias("desvio_padrao"),
            F.countDistinct(coluna).alias("valores_distintos"),
            F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0)).alias("nulos"),
            F.count("*").alias("total"),
        )
        .collect()[0]
    )

    resultados_variabilidade.append(
        (
            coluna,
            estatisticas["min"],
            estatisticas["p05"],
            estatisticas["p25"],
            estatisticas["mediana"],
            estatisticas["p75"],
            estatisticas["p95"],
            estatisticas["max"],
            estatisticas["desvio_padrao"],
            estatisticas["valores_distintos"],
            estatisticas["nulos"],
            estatisticas["nulos"] / estatisticas["total"],
        )
    )

In [0]:
colunas_resultado = [
    "feature",
    "min",
    "p05",
    "p25",
    "mediana",
    "p75",
    "p95",
    "max",
    "desvio_padrao",
    "valores_distintos",
    "nulos",
    "pct_nulos",
]

df_variabilidade = spark.createDataFrame(resultados_variabilidade, colunas_resultado)

display(df_variabilidade)

## 4.2 Relação com o Target
Nesta etapa é avaliada a associação individual entre cada feature numérica e o target de alfabetização.

A análise utiliza três perspectivas complementares:

- correlação linear com o target;
- ROC-AUC utilizando cada variável isoladamente como score;
- diferença da taxa de alfabetização ao longo da distribuição da feature.

Função 

In [0]:
def medir_associacao_univariada(df, features, split_nome):
    resultados = []

    evaluator = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="score", metricName="areaUnderROC"
    )

    df_base = df.withColumn("label", F.col("alfabetizado").cast("double"))

    for feature in features:
        df_feature = df_base.filter(F.col(feature).isNotNull()).select(
            "label", F.col(feature).cast("double").alias("score")
        )

        correlacao = df_feature.agg(F.corr("score", "label").alias("corr")).first()[
            "corr"
        ]

        auc = evaluator.evaluate(df_feature)

        # Como uma feature pode ter relação inversa com o target,
        # transformamos o AUC em capacidade discriminatória independente
        # da direção.
        auc_discriminatorio = max(auc, 1 - auc)

        resultados.append((split_nome, feature, correlacao, auc, auc_discriminatorio))

    return resultados

Execução

In [0]:
features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
]

resultados_associacao = []

resultados_associacao += medir_associacao_univariada(
    df_desenvolvimento.filter(F.col("split") == "treino"), features_numericas, "treino"
)

resultados_associacao += medir_associacao_univariada(
    df_desenvolvimento.filter(F.col("split") == "validacao"),
    features_numericas,
    "validacao",
)

Resultado

In [0]:
df_associacao = spark.createDataFrame(
    resultados_associacao,
    ["split", "feature", "correlacao", "roc_auc", "roc_auc_discriminatorio"],
)

display(df_associacao.orderBy("split", F.desc("roc_auc_discriminatorio")))

## 4.3 Efeito não linear por decis

In [0]:
def analisar_decis_feature(df, feature):

    df_feature = (
        df
        .filter(F.col(feature).isNotNull())
    )

    discretizer = QuantileDiscretizer(
        numBuckets=10,
        inputCol=feature,
        outputCol="decil",
        handleInvalid="skip"
    )

    modelo_discretizer = discretizer.fit(df_feature)

    return (
        modelo_discretizer
        .transform(df_feature)
        .groupBy("decil")
        .agg(
            F.count("*").alias("qtd_alunos"),
            F.avg(feature).alias("media_feature"),
            F.avg(
                F.col("alfabetizado").cast("double")
            ).alias("taxa_alfabetizacao")
        )
        .orderBy("decil")
    )

In [0]:
resultados_decis = []

df_treino_diag = df_desenvolvimento.filter(F.col("split") == "treino")

for feature in features_numericas:
    df_decis = analisar_decis_feature(df_treino_diag, feature)

    resumo = df_decis.agg(
        F.min("taxa_alfabetizacao").alias("min_taxa"),
        F.max("taxa_alfabetizacao").alias("max_taxa"),
    ).first()

    resultados_decis.append(
        (
            feature,
            resumo["min_taxa"],
            resumo["max_taxa"],
            resumo["max_taxa"] - resumo["min_taxa"],
        )
    )

In [0]:
df_amplitude_decis = spark.createDataFrame(
    resultados_decis,
    ["feature", "min_taxa_alfabetizacao", "max_taxa_alfabetizacao", "amplitude_taxa"],
)

display(df_amplitude_decis.orderBy(F.desc("amplitude_taxa")))

## 4.4 Correlação Variáveis

In [0]:
correlacoes = []

df_treino_diag = df_desenvolvimento.filter(F.col("split") == "treino")

for i, feature_1 in enumerate(features_numericas):
    for feature_2 in features_numericas[i + 1 :]:
        corr = df_treino_diag.select(
            F.corr(feature_1, feature_2).alias("correlacao")
        ).first()["correlacao"]

        correlacoes.append(
            (feature_1, feature_2, corr, abs(corr) if corr is not None else None)
        )

In [0]:
df_correlacoes_features = spark.createDataFrame(
    correlacoes, ["feature_1", "feature_2", "correlacao", "correlacao_abs"]
)

display(df_correlacoes_features.orderBy(F.desc("correlacao_abs")))

## Diagnóstico do Poder Discriminatório Individual

As análises univariadas indicaram baixo poder discriminatório individual das features atualmente disponíveis.

Nenhuma variável apresentou ROC-AUC superior a aproximadamente 0,57 em treino ou validação, e as correlações individuais com o target permaneceram abaixo de 0,11 em valor absoluto.

INSE, quantidade relativa de docentes, acesso à internet para aprendizagem e biblioteca/sala de leitura apresentaram os sinais mais relevantes.

A análise de correlação identificou relações moderadas entre algumas variáveis, mas não foram observados níveis de correlação suficientemente elevados para caracterizar redundância extrema.

As maiores associações ocorreram entre INSE e PIB per capita (0,656), e entre as razões de docentes e turmas por 100 alunos (0,630).

Os resultados reforçam a hipótese de que as features atuais conseguem caracterizar diferenças médias entre contextos educacionais, mas possuem capacidade limitada para distinguir individualmente alunos alfabetizados e não alfabetizados.

Os resultados indicam que o principal gargalo da primeira rodada de modelagem está no poder discriminatório das informações disponíveis, e não apenas na complexidade dos algoritmos utilizados.